# Data Audit in Pandas: Waiting or Paying?

Team: The Foreigners. Part B of Submission 4, written as code.

The project asks why some people in Ireland do not get the medical care they need, because of waiting or because of cost. It uses two sources, and this notebook audits both. For each one we loaded the raw data, checked missing values, duplicates and unusual values, cleaned it, and built one small prepared table for the charts.

| # | Source | Prepared table | What it shows |
| --- | --- | --- | --- |
| 1 | Eurostat survey on unmet need for medical care | `unmet_need_by_reason` | How many people said they went without care because of cost or waiting, Ireland, the EU average and the other EU countries |
| 2 | NTPF outpatient waiting list | `waiting_list_national` | How many people are waiting for a hospital outpatient appointment, month by month |

The Eurostat data is downloaded when the notebook runs. NTPF does not allow downloads from Python, so its nine files were saved from the NTPF website and are read from `ntpf_files_for_data_raw.zip`.

In [1]:
# Colab only: open the project folder on Google Drive (does nothing elsewhere)
import os

try:
    from google.colab import drive

    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/Colab Notebooks/DataVis - Team Project Stuff/S4_data_audit")
except ImportError:
    pass

## 0. Setup

We import the tools we need, set the folders for the raw and prepared data, and start an empty list that records every file we load.

In [2]:
import zipfile
from pathlib import Path

%pip install pandas
import pandas as pd

ACCESS_DATE = "2026-11-11"  # the date the audited numbers were pulled
RAW = Path("data/raw")  # raw files, kept unchanged
PREPARED = Path("data/prepared")  # prepared tables
for folder in ["eurostat", "ntpf"]:
    (RAW / folder).mkdir(parents=True, exist_ok=True)
PREPARED.mkdir(parents=True, exist_ok=True)

sources = []  # a small log of what was loaded: name, where from, rows


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: C:\Users\teacu\AppData\Local\Temp\claude\b299daf126cec770\733c5b63-9f0c-46a9-a8f0-ff2eafa4879b\scratchpad\venv\Scripts\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 1. Eurostat: unmet need for medical care

The EU-SILC survey asks people whether there was a time in the last 12 months when they needed a medical examination but did not get one, and why. We kept two reasons: too expensive (`TXP`) and waiting list (`WLIST`). We kept all ages (16 and over) and the total for sex and income. We downloaded every area Eurostat has for these two reasons, then kept the 27 EU countries and the EU27 average (`EU27_2020`), for 2016 to 2025. One row in the prepared table is one area, year and reason.

Eurostat sends the data wide, with one column per year. A missing value is `:` and a flag is a letter after the number, for example `1.0 b`.

**Why we need this table.** It is the core of our questions about why people go without care. It shows, year by year, how many people said the reason was cost and how many said waiting, for Ireland, the EU average and the other EU countries, so we can see whether waiting has replaced cost, whether Ireland differs from the EU average, and where Ireland sits among the EU countries.

### 1.1 Load the raw data

We download the survey table from Eurostat for the two reasons, save an untouched copy in `data/raw`, and show its size. The table comes back wide, with one column per year.

In [3]:
EUROSTAT = "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/"
url = (
    EUROSTAT
    + "hlth_silc_08/A.PC.TOTAL.TXP+WLIST.Y_GE16.T.?format=TSV&compressed=false&startPeriod=2016&endPeriod=2025"
)

raw = pd.read_csv(url, sep="\t")
raw.columns = [c.strip() for c in raw.columns]
raw.to_csv(RAW / "eurostat" / "hlth_silc_08.csv", index=False)  # keep the raw table
sources.append(("Eurostat hlth_silc_08", url, len(raw)))

print(raw.shape)
raw.head()

(92, 11)


,"freq,unit,quant_inc,reason,age,sex,geo\TIME_PERIOD",2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,"A,PC,TOTAL,TXP,Y_GE16,T,AL",:,12.5,13.7,13.5,9.1,9.2,8.8,8.6,8.0,7.7
1,"A,PC,TOTAL,TXP,Y_GE16,T,AT",0.2,0.2,0.1,0.2,0.1,0.1,0.2,0.2,0.2,0.3
2,"A,PC,TOTAL,TXP,Y_GE16,T,BE",2.2,2.0,1.8,1.8 b,1.4,1.2,0.9,0.9,1.1,0.8
3,"A,PC,TOTAL,TXP,Y_GE16,T,BG",2.2,1.6,1.5,1.1,1.0,0.7,0.9,0.7,0.8,0.7
4,"A,PC,TOTAL,TXP,Y_GE16,T,CH",0.5,0.4,0.6,0.6,0.5,0.5,0.8,1.2,1.2,:


The raw table has 92 rows (46 area codes by 2 reasons) and one column per year. The first column holds the codes: frequency, unit, income, reason, age, sex and area. Besides the 27 EU countries, the area codes include the EU27 average, older EU and euro area totals (such as `EU28`), the UK and other non-EU countries.

### 1.2 Make it one row per year

We split the first column into separate code columns (reason, age, area and so on) and turn the year columns into rows. Each cell holds a number and sometimes a flag letter, so we split those into `pct` and `flag`. A `:` means missing and becomes empty.

In [4]:
first = raw.columns[0]
codes = raw[first].str.split(",", expand=True)
codes.columns = ["freq", "unit", "income", "reason", "age", "sex", "geo"]
raw = pd.concat([codes, raw.drop(columns=first)], axis=1)

years = [c for c in raw.columns if c.isdigit()]
long = raw.melt(id_vars=codes.columns, value_vars=years, var_name="year", value_name="cell")

cell = long["cell"].astype(str).str.strip().str.extract(r"^(\S*)\s*(.*)$")  # the number, then the flag
long["pct"] = pd.to_numeric(cell[0], errors="coerce")  # ":" becomes NaN
long["flag"] = cell[1]
long["year"] = long["year"].astype(int)

long[["geo", "year", "reason", "pct", "flag"]].head()

,geo,year,reason,pct,flag
0,AL,2016,TXP,NaN,
1,AT,2016,TXP,0.2,
2,BE,2016,TXP,2.2,
3,BG,2016,TXP,2.2,
4,CH,2016,TXP,0.5,


### 1.3 Choose the areas

We keep only the 27 EU countries and the EU27 average, and list the area codes we dropped. The `assert` lines stop the notebook if a count is not what we expect.

In [5]:
EU27 = "AT BE BG HR CY CZ DK EE FI FR DE EL HU IE IT LV LT LU MT NL PL PT RO SK SI ES SE".split()
kept = long[long["geo"].isin(EU27 + ["EU27_2020"])].copy()  # the 27 EU countries and the EU27 average
dropped = sorted(set(long["geo"]) - set(kept["geo"]))

print(
    "Area codes loaded:", long["geo"].nunique(), "| kept:", kept["geo"].nunique(), "| dropped:", len(dropped)
)
print("Dropped:", dropped)

assert long["geo"].nunique() == 46 and kept["geo"].nunique() == 28
assert len(long) == 920 and len(kept) == 560

Area codes loaded: 46 | kept: 28 | dropped: 18
Dropped: ['AL', 'CH', 'EA', 'EA18', 'EA19', 'EA20', 'EA21', 'EU', 'EU27_2007', 'EU28', 'IS', 'ME', 'MK', 'NO', 'RS', 'TR', 'UK', 'XK']


**What we did.** Eurostat lists 46 area codes. We kept the 27 EU countries and the EU27 average (28 areas, 560 values) and dropped 18 codes: the UK, Switzerland, Norway and other non-EU countries, and older EU and euro area totals, which would count the same countries twice.

### 1.4 Missing values and duplicates

We count the empty values and the repeated rows, and show which values are missing.

In [6]:
missing = kept["pct"].isna().sum()
gaps = kept[kept["pct"].isna()][["geo", "year", "reason"]]
duplicates = kept.duplicated(["geo", "year", "reason"]).sum()
print("Rows:", len(kept), "| missing:", missing, "| duplicates:", duplicates)
print(gaps)

assert missing == 2
assert set(gaps["geo"]) == {"IT"} and set(gaps["year"]) == {2020}
assert duplicates == 0

Rows: 560 | missing: 2 | duplicates: 0
    geo  year reason
395  IT  2020    TXP
441  IT  2020  WLIST


**What we found.** There are 560 values (28 areas, 2 reasons, 10 years). Two are missing, both for Italy in 2020 (cost and waiting): Eurostat gives no value, so we left them blank. None are repeated.

### 1.5 Unusual values

We list Eurostat's quality flags, find the break flags and the highest value, and look at Ireland's numbers and its rank among the 27 countries. The `assert` lines confirm each number.

In [7]:
flagged = kept[kept["flag"] != ""]
breaks = kept[kept["flag"].str.contains("b")]
print("Flags:", flagged["flag"].value_counts().to_dict())
print("Break flags:", len(breaks), "in", breaks["geo"].nunique(), "areas:", sorted(breaks["geo"].unique()))
print("Ireland flagged:", flagged[flagged["geo"] == "IE"][["year", "reason", "flag"]].values.tolist())

top = kept.nlargest(1, "pct").iloc[0]
print("Highest value:", top["geo"], top["year"], top["reason"], top["pct"])

ie = kept[kept["geo"] == "IE"].pivot(index="year", columns="reason", values="pct")
eu = kept[kept["geo"] == "EU27_2020"].pivot(index="year", columns="reason", values="pct")
print("Ireland waiting 2016 and 2025:", ie.loc[2016, "WLIST"], ie.loc[2025, "WLIST"])
print("Ireland cost 2016 and 2025:", ie.loc[2016, "TXP"], ie.loc[2025, "TXP"])
print("EU cost 2016 to 2018:", eu.loc[2016:2018, "TXP"].tolist())

last = kept[(kept["year"] == 2025) & kept["geo"].isin(EU27)]
rank = last.groupby("reason")["pct"].rank(ascending=False, method="first")
ie_rank = rank[last["geo"] == "IE"]
ie_rank.index = last.loc[ie_rank.index, "reason"]
wide = last.pivot(index="geo", columns="reason", values="pct")
print("Ireland rank in 2025 (1 = highest of 27):", ie_rank.to_dict())
print("Countries where waiting is above cost in 2025:", (wide["WLIST"] > wide["TXP"]).sum())

assert flagged["flag"].value_counts().to_dict() == {"n": 45, "b": 25, "p": 4, "e": 2}
assert len(breaks) == 25 and breaks["geo"].nunique() == 8
assert [r[0] for r in flagged[flagged["geo"] == "IE"][["year"]].values.tolist()] == [2019, 2019, 2020, 2020]
assert (top["geo"], top["year"], top["reason"], top["pct"]) == ("EE", 2018, "WLIST", 15.0)
assert (ie.loc[2016, "WLIST"], ie.loc[2025, "WLIST"]) == (1.0, 2.1)
assert (ie.loc[2016, "TXP"], ie.loc[2025, "TXP"]) == (1.5, 0.6)
assert eu.loc[2016:2018, "TXP"].tolist() == [1.9, 1.0, 1.1]
assert ie_rank.to_dict() == {"TXP": 9, "WLIST": 8}
assert (wide["WLIST"] > wide["TXP"]).sum() == 19

Flags:

 {'n': 45, 'b': 25, 'p': 4, 'e': 2}
Break flags: 25 in 8 areas: ['BE', 'DE', 'FR', 'HR', 'HU', 'IE', 'LU', 'PL']
Ireland flagged: [[2019, 'TXP', 'b'], [2019, 'WLIST', 'b'], [2020, 'TXP', 'b'], [2020, 'WLIST', 'b']]
Highest value: EE 2018 WLIST 15.0
Ireland waiting 2016 and 2025: 1.0 2.1
Ireland cost 2016 and 2025: 1.5 0.6
EU cost 2016 to 2018: [1.9, 1.0, 1.1]
Ireland rank in 2025 (1 = highest of 27): {'TXP': 9.0, 'WLIST': 8.0}
Countries where waiting is above cost in 2025: 19


**What we found.** Eurostat's quality flags appear on 76 values: `n` (45, not significant), `b` (25, break in the series), `p` (4, provisional) and `e` (2, estimated). The 25 break flags are in 8 areas (Belgium, Germany, France, Croatia, Hungary, Ireland, Luxembourg and Poland). Ireland's four are in 2019 and 2020 and mean the survey method changed, so those years are not fully comparable with the others. Some values are large: the highest is Estonia's waiting answer in 2018 (15.0%), far above Ireland. The EU average cost value for 2016 (1.9%) is much higher than 2017 (1.0%), with no flag and no explanation that we could find. Ireland's waiting reason rises from 1.0% in 2016 to 2.1% in 2025, while its cost reason falls from 1.5% to 0.6%. In 2025 Ireland had the 8th highest waiting answer and the 9th highest cost answer of the 27 EU countries, and waiting was above cost in 19 of the 27. Nothing was removed.

### 1.6 Clean and build the prepared table

We build the prepared table: rename the columns, add readable country and reason names, and add a `group` column (Ireland, EU average or other EU country).

We also add `is_break`. It is `True` when the value carries Eurostat's flag `b`, which means a break in the series: Eurostat changed how it measured, so that value is not fully comparable with the other years. We keep these rows and use the column to mark them on the charts. Ireland's 2019 and 2020 values are the ones flagged.

In [8]:
names = {
    "AT": "Austria",
    "BE": "Belgium",
    "BG": "Bulgaria",
    "HR": "Croatia",
    "CY": "Cyprus",
    "CZ": "Czechia",
    "DK": "Denmark",
    "EE": "Estonia",
    "FI": "Finland",
    "FR": "France",
    "DE": "Germany",
    "EL": "Greece",
    "HU": "Hungary",
    "IE": "Ireland",
    "IT": "Italy",
    "LV": "Latvia",
    "LT": "Lithuania",
    "LU": "Luxembourg",
    "MT": "Malta",
    "NL": "Netherlands",
    "PL": "Poland",
    "PT": "Portugal",
    "RO": "Romania",
    "SK": "Slovakia",
    "SI": "Slovenia",
    "ES": "Spain",
    "SE": "Sweden",
    "EU27_2020": "EU27 average",
}

unmet_need_by_reason = kept.rename(columns={"geo": "area_code"})
unmet_need_by_reason["area"] = unmet_need_by_reason["area_code"].map(names)
unmet_need_by_reason["group"] = "Other EU country"
unmet_need_by_reason.loc[unmet_need_by_reason["area_code"] == "IE", "group"] = "Ireland"
unmet_need_by_reason.loc[unmet_need_by_reason["area_code"] == "EU27_2020", "group"] = "EU average"
unmet_need_by_reason["reason"] = unmet_need_by_reason["reason"].map(
    {"TXP": "Too expensive", "WLIST": "Waiting list"}
)
unmet_need_by_reason["is_break"] = unmet_need_by_reason["flag"].str.contains("b")

columns = ["area", "area_code", "group", "year", "reason", "pct", "flag", "is_break"]
unmet_need_by_reason = unmet_need_by_reason[columns]
unmet_need_by_reason = unmet_need_by_reason.sort_values(["area", "reason", "year"]).reset_index(drop=True)

unmet_need_by_reason.head()

,area,area_code,group,year,reason,pct,flag,is_break
0,Austria,AT,Other EU country,2016,Too expensive,0.2,,False
1,Austria,AT,Other EU country,2017,Too expensive,0.2,,False
2,Austria,AT,Other EU country,2018,Too expensive,0.1,,False
3,Austria,AT,Other EU country,2019,Too expensive,0.2,,False
4,Austria,AT,Other EU country,2020,Too expensive,0.1,,False


We check the prepared table (row count, no repeated rows, 28 areas, 25 break values) and count the rows for each group and reason, so we can see that both reasons are there for every group.

In [9]:
assert len(unmet_need_by_reason) == 560
assert not unmet_need_by_reason.duplicated(["area", "year", "reason"]).any()
assert unmet_need_by_reason["area"].nunique() == 28
assert unmet_need_by_reason["is_break"].sum() == 25
assert round(unmet_need_by_reason["pct"].sum(), 1) == 701.0

unmet_need_by_reason.groupby(["group", "reason"]).size()

group             reason       
EU average        Too expensive     10
                  Waiting list      10
Ireland           Too expensive     10
                  Waiting list      10
Other EU country  Too expensive    260
                  Waiting list     260
dtype: int64

**What we did.** Cleaning: kept the two reasons, all ages, the 27 EU countries and the EU average, and 2016 to 2025, and dropped the 18 other area codes. Transformations: country names, a `group` column (Ireland, EU average or other EU country), readable reason names, and a column `is_break` that marks the 25 break-flagged values. Aggregation: none, because Eurostat already gives yearly percentages. Missing values (Italy 2020) stay blank. The values are percentages of all people aged 16 and over, so they are small and each point can move by a few tenths from survey noise alone.

## 2. NTPF: outpatient waiting list

The National Treatment Purchase Fund (NTPF) publishes a monthly snapshot of the people waiting for a first outpatient appointment at a public hospital. The files come in two layouts, so we worked with two groups:

| Group | Files | Snapshots | One row is | Waiting-time columns |
| --- | --- | --- | --- | --- |
| By hospital | 3 | 27, Jan 2019 to Mar 2021 | snapshot, hospital, specialty, age profile, time band | 3-month bands, in one column |
| By specialty | 6 | 65, Apr 2021 to Sep 2026 | snapshot, adult or child, specialty | four 6-month bands, one column each |

The project does not split by age, so we only need the national total of people waiting, by how long they have waited. The prepared table has one row per snapshot and waiting-time band.

**Why we need this table.** The survey says what people report, but not how the hospital list itself changed. This table gives the actual number of people waiting each month, and for how long, so we can check whether the list moved in the same direction as the survey answers.

### 2.1 Load the files

We list the nine NTPF files, unzip them into `data/raw/ntpf`, read each one as text, and check that each has the number of rows we expect.

In [10]:
hospital_files = [
    "OP-Waiting-List-by-Group-Hospital-2019.csv",
    "OP-Waiting-List-by-Group-Hospital-2020.csv",
    "OP-Waiting-List-By-Group-Hospital-2021-1.csv",
]
specialty_files = [
    "OpenData_OPNational02_2021-1.csv",
    "OpenData_OPNational02_2022-1.csv",
    "OpenData_OPNational02_2023-1.csv",
    "OpenData_OPNational02_2024-1.csv",
    "OpenData_OPNational02_2025.csv",
    "OpenData_OPNational02_2026.csv",
]

if Path("ntpf_files_for_data_raw.zip").exists():  # the team zip holds the nine files
    with zipfile.ZipFile("ntpf_files_for_data_raw.zip") as z:
        for member in z.namelist():
            if member.lower().endswith(".csv"):
                name = member.replace("\\", "/").split("/")[-1]
                (RAW / "ntpf" / name).write_bytes(z.read(member))
elif not (RAW / "ntpf" / hospital_files[0]).exists():
    raise FileNotFoundError("Put ntpf_files_for_data_raw.zip in the same folder as this notebook.")

ntpf = {}
for f in hospital_files + specialty_files:
    ntpf[f] = pd.read_csv(RAW / "ntpf" / f, encoding="utf-8-sig", dtype=str)  # read as text first
    sources.append((f, "https://www.ntpf.ie/waiting-list-data/open-data/", len(ntpf[f])))

print([len(t) for t in ntpf.values()])
assert [len(t) for t in ntpf.values()] == [80236, 85273, 22521, 747, 1056, 1011, 934, 975, 709]

[80236, 85273, 22521, 747, 1056, 1011, 934, 975, 709]


We read everything as text first because the numbers can contain commas, and then converted them ourselves.

### 2.2 Hospital files (Jan 2019 to Mar 2021)

We stack the three hospital files and turn the date and the counts into proper types. Then we check for missing counts, missing waiting bands, repeated rows and the "Small Volume" rows.

In [11]:
hospital = pd.concat([ntpf[f] for f in hospital_files], ignore_index=True)
hospital["snapshot"] = pd.to_datetime(hospital["Archive_Date"], format="%d/%m/%Y")
hospital["people"] = pd.to_numeric(hospital["Total"].str.replace(",", ""), errors="coerce")
hospital["time_band"] = hospital["Time_Bands"].str.strip()

print("Rows:", len(hospital), "| snapshots:", hospital["snapshot"].nunique())
print(
    "Missing people:",
    hospital["people"].isna().sum(),
    "| missing time band:",
    hospital["time_band"].isna().sum(),
)
print("Repeated rows:", hospital.duplicated().sum())
print(hospital["time_band"].value_counts())

small = hospital["Speciality"].str.contains("Small Volume").sum()
print("'Small Volume' rows:", small)

assert hospital["snapshot"].nunique() == 27
assert hospital["people"].notna().all()
assert hospital["time_band"].isna().sum() == 2
assert hospital.duplicated().sum() == 0

Rows: 188030 | snapshots: 27
Missing people: 0 | missing time band: 2


Repeated rows: 0
time_band
0-3 Months      36773
3-6 Months      32745
6-9 Months      29598
9-12 Months     26408
12-15 Months    23200
15-18 Months    20121
18 Months +     19183
Name: count, dtype: int64
'Small Volume' rows: 611


**What we found.** There are 27 monthly snapshots and no missing counts. Two rows have no time band (2 people in total), and no rows are repeated. Some rows are called "Small Volume": NTPF combines groups with fewer than 20 people for privacy. We kept them, because removing them would lower the totals.

### 2.3 Specialty files (Apr 2021 to Sep 2026)

We stack the six specialty files and turn the date and the four waiting-band columns into numbers. Then we check for missing numbers and compare the bands with NTPF's own total.

In [12]:
specialty = pd.concat([ntpf[f] for f in specialty_files], ignore_index=True)
specialty["snapshot"] = pd.to_datetime(specialty["ArchiveDate"], format="%d/%m/%Y")

bands = ["0-6 Months", "6-12 Months", "12-18 Months", "18 Months +"]
for col in bands + ["Total"]:
    specialty[col] = pd.to_numeric(specialty[col].str.replace(",", ""), errors="coerce")

print("Rows:", len(specialty), "| snapshots:", specialty["snapshot"].nunique())
print("Missing numbers:", specialty[bands + ["Total"]].isna().sum().sum())
print("Four bands add up to:", specialty[bands].sum().sum(), "| Total column:", specialty["Total"].sum())

assert specialty["snapshot"].nunique() == 65
assert specialty[bands + ["Total"]].notna().all().all()

Rows: 5432 | snapshots: 65
Missing numbers: 0
Four bands add up to: 39903505 | Total column: 39903398


**What we found.** There are 65 snapshots and no missing numbers. The four bands add up to 39,903,505 people over all rows, and NTPF's own `Total` column adds up to 39,903,398, a difference of 107. We built the table from the four bands and left NTPF's numbers as they are.

### 2.4 Repeated rows in 2025 and 2026

We look for rows that share the same date, adult or child and specialty, which should be unique. The table compares the total when all rows are added with the total when only the first copy is kept, next to the snapshots before and after.

In [13]:
key = ["snapshot", "Adult_Child", "Speciality"]
repeated = specialty[specialty.duplicated(key, keep=False)]
days = sorted(repeated["snapshot"].unique())
print(
    "Extra rows:", specialty.duplicated(key).sum(), "| snapshots with repeats:", [str(d)[:10] for d in days]
)

all_rows = specialty.groupby("snapshot")["Total"].sum()
first_copy = specialty.drop_duplicates(key).groupby("snapshot")["Total"].sum()
check = pd.DataFrame(
    {
        "snapshot before": all_rows.shift(1),
        "add all rows": all_rows,
        "first copy only": first_copy,
        "snapshot after": all_rows.shift(-1),
    }
)
check.loc[days].astype(int)

Extra rows: 85 | snapshots with repeats: ['2025-10-30', '2025-11-27', '2026-01-29']


,snapshot before,add all rows,first copy only,snapshot after
snapshot,,,,
2025-10-30,626525,626955,379661,623818
2025-11-27,626955,623818,384031,611989
2026-01-29,611989,631607,458457,642161


**What we found.** On three snapshots (30 Oct 2025, 27 Nov 2025 and 29 Jan 2026) the same adult or child and specialty appears more than once with different numbers, 85 extra rows in total. If we add all the rows, the total sits between the snapshots before and after. If we keep only the first copy, the total drops by about 170,000 to 250,000 people, which is not believable. So we treated them as split rows and added them together. We have not yet confirmed this with NTPF.

### 2.5 Build the national table

We make the 3-month bands match the 6-month bands, stack the hospital and specialty data into one list of snapshot, band and people, and add it up to one row per snapshot and band.

In [14]:
band_map = {
    "0-3 Months": "0-6 months",
    "3-6 Months": "0-6 months",
    "6-9 Months": "6-12 months",
    "9-12 Months": "6-12 months",
    "12-15 Months": "12-18 months",
    "15-18 Months": "12-18 months",
    "18 Months +": "18+ months",
}
hospital["band"] = hospital["time_band"].map(band_map).fillna("unknown")  # 3-month bands become 6-month bands
part1 = hospital[["snapshot", "band", "people"]]

part2 = specialty.melt(id_vars="snapshot", value_vars=bands, var_name="band", value_name="people")
part2["band"] = part2["band"].map(
    {
        "0-6 Months": "0-6 months",
        "6-12 Months": "6-12 months",
        "12-18 Months": "12-18 months",
        "18 Months +": "18+ months",
    }
)

waiting_list_national = pd.concat([part1, part2], ignore_index=True)
waiting_list_national = waiting_list_national.groupby(["snapshot", "band"], as_index=False)["people"].sum()

print(waiting_list_national.shape)
waiting_list_national.head()

(370, 3)

,snapshot,band,people
0,2019-01-31,0-6 months,246063
1,2019-01-31,12-18 months,62170
2,2019-01-31,18+ months,96243
3,2019-01-31,6-12 months,118749
4,2019-02-28,0-6 months,258569


The hospital files use seven 3-month bands and the specialty files use four 6-month bands, so the 3-month bands were combined into 6-month bands to match. The two groups were then stacked and added up to one row per snapshot and band.

### 2.6 Unusual values and missing months

We add up each snapshot to a national total, find the lowest and highest, check that the two file groups join up in 2021, list the yearly averages, and look for months with no snapshot.

In [15]:
totals = waiting_list_national.groupby("snapshot")["people"].sum()
low, high = totals.idxmin(), totals.idxmax()
print("Lowest:", low.date(), totals.min(), "| highest:", high.date(), totals.max())
print(
    "25 Mar 2021 (hospital files):",
    totals["2021-03-25"],
    "| 29 Apr 2021 (specialty files):",
    totals["2021-04-29"],
)

yearly = totals.groupby(totals.index.year).agg(["mean", "count"]).round(0).astype(int)
print(yearly)

months = pd.period_range(totals.index.min(), totals.index.max(), freq="M")
no_snapshot = [str(m) for m in months if m not in set(totals.index.to_period("M"))]
print(len(totals), "snapshots in", len(months), "months | no snapshot in:", no_snapshot)

Lowest: 2019-01-31 523225 | highest: 2026-09-24 692529
25 Mar 2021 (hospital files): 628756 | 29 Apr 2021 (specialty files): 630319
            mean  count
snapshot               
2019      555628     12
2020      588473     12
2021      636015     11
2022      619584     12
2023      592166     12
2024      585630     12
2025      602124     12
2026      665768      9


92 snapshots in 93 months | no snapshot in: ['2021-06']


We check that the new table holds the same number of people as the two inputs, has no repeated rows, has 92 snapshots with only June 2021 missing, and has the expected lowest, highest and yearly values.

In [16]:
assert waiting_list_national["people"].sum() == hospital["people"].sum() + specialty[bands].sum().sum()
assert not waiting_list_national.duplicated(["snapshot", "band"]).any()
assert len(totals) == 92 and no_snapshot == ["2021-06"]
assert (totals.min(), str(low.date())) == (523225, "2019-01-31")
assert (totals.max(), str(high.date())) == (692529, "2026-09-24")
assert yearly.loc[2019, "mean"] == 555628 and yearly.loc[2026, "mean"] == 665768

**What we found.** The national list ran from 523,225 people (31 Jan 2019) to 692,529 people (24 Sep 2026). The yearly average was about 555,600 in 2019 and about 665,800 in 2026 so far (9 snapshots). The two file groups join up: 628,756 people on 25 Mar 2021 and 630,319 on 29 Apr 2021. There are 92 snapshots in 93 months, because June 2021 has no snapshot, and we do not know why. The 2026 average only covers January to September, so it is not a full year.

## 3. Summary and export

Two prepared tables came out of the two sources.

The two tables answer different questions. `unmet_need_by_reason` says why people go without care, as a yearly percentage for Ireland, the EU average and the other EU countries. `waiting_list_national` says how many people are waiting for a hospital outpatient appointment, as a monthly count for Ireland.

We put the two prepared tables in one list, check that each has one row per key, and show a summary with the number of rows and the type of each column.

In [17]:
prepared = {"unmet_need_by_reason": unmet_need_by_reason, "waiting_list_national": waiting_list_national}
keys = {"unmet_need_by_reason": ["area", "year", "reason"], "waiting_list_national": ["snapshot", "band"]}

for name, key_columns in keys.items():
    assert not prepared[name].duplicated(key_columns).any(), f"{name} has more than one row per key"

types = pd.DataFrame(
    [
        [
            "unmet_need_by_reason",
            len(unmet_need_by_reason),
            "area, year, reason",
            "pct",
            "",
            "area_code, group, reason, flag, is_break",
            "year",
            "area",
        ],
        [
            "waiting_list_national",
            len(waiting_list_national),
            "snapshot, band",
            "people",
            "band",
            "",
            "snapshot",
            "",
        ],
    ],
    columns=["table", "rows", "one row per", "Q", "O", "N", "T", "G"],
)
types

,table,rows,one row per,Q,O,N,T,G
0,unmet_need_by_reason,560,"area, year, reason",pct,,"area_code, group, reason, flag, is_break",year,area
1,waiting_list_national,370,"snapshot, band",people,band,,snapshot,


The columns are marked Q quantitative, O ordinal, N nominal, T temporal and G geographical. The two tables share no column, so we will compare them side by side, year by year, in the charts and not join them.

We save both prepared tables as CSV files, together with the log of the files we loaded.

In [18]:
for name, table in prepared.items():
    table.to_csv(PREPARED / f"{name}_v1_{ACCESS_DATE}.csv", index=False)

log = pd.DataFrame(sources, columns=["source", "where", "rows"]).assign(accessed=ACCESS_DATE)
log.to_csv(Path("data") / f"provenance_log_{ACCESS_DATE}.csv", index=False)
print("Saved", len(prepared), "prepared tables and a log of", len(log), "files.")

Saved 2 prepared tables and a log of 10 files.


### Audit at a glance

| Source | Missing | Duplicates | Unusual values | Main assumption |
| --- | --- | --- | --- | --- |
| Eurostat unmet need | 2 of 560 (Italy, 2020) | None | 76 flagged values, 25 of them breaks in 8 areas (Ireland in 2019 and 2020). Estonia reaches 15.0%. EU average cost value for 2016 looks high | The values are small, self-reported percentages |
| NTPF by hospital (2019 to Mar 2021) | 2 rows with no time band (2 people) | None | "Small Volume" privacy groups | 3-month bands can be combined into 6-month bands |
| NTPF by specialty (Apr 2021 on) | None | 85 repeated rows in 2025 and 2026 | June 2021 has no snapshot | The repeated rows are split rows and are added together |

